In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/despesas_MA_2025_pagamentos.csv", sep=",")

C:\Users\edils\AppData\Local\Temp\ipykernel_23896\1421566489.py:1: DtypeWarning: Columns (9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/despesas_MA_2025_pagamentos.csv", sep=",")


In [3]:
print(df.dtypes)

ano                    int64
mes                    int64
fase                  object
codigo_unidade         int64
unidade               object
valor                float64
codigo_credor         object
credor_nome           object
codigo_funcao        float64
codigo_natureza       object
cod_grupo_despesa     object
dtype: object


In [4]:
qtd_linhas , qtd_colunas = df.shape
print(f"Quantidade de {qtd_linhas} linhas \nQuantidade de {qtd_colunas} colunas")

Quantidade de 506460 linhas 
Quantidade de 11 colunas


In [5]:
df_colunas = df.columns
for colunas in df_colunas:
    print(f"Coluna - {colunas}")

Coluna - ano
Coluna - mes
Coluna - fase
Coluna - codigo_unidade
Coluna - unidade
Coluna - valor
Coluna - codigo_credor
Coluna - credor_nome
Coluna - codigo_funcao
Coluna - codigo_natureza
Coluna - cod_grupo_despesa


In [6]:
print(f"Total de linhas {df['valor'].isna().sum()} nulas")

Total de linhas 0 nulas


In [7]:
df["fase"].value_counts()

fase
PAGAMENTO    506460
Name: count, dtype: int64

In [8]:
valor_minimo = df["valor"].min()
valor_maximo = df["valor"].max()

print(f"Valor Minino: {valor_minimo}\nValor Máximo: {valor_maximo}")

Valor Minino: 1.0
Valor Máximo: 34913036878.0


In [9]:
pagamento_menor_10 = df[df["valor"]<=1000]
display(pagamento_menor_10["valor"].value_counts())

valor
600.0    3391
700.0    1931
900.0    1320
500.0    1279
300.0    1105
         ... 
914.0       1
722.0       1
363.0       1
124.0       1
654.0       1
Name: count, Length: 898, dtype: int64

In [10]:
pagamento_maior_max = df[df["valor"]>=1913036878.0]

In [11]:
display(pagamento_maior_max["valor"].value_counts().head(6))

valor
2.801974e+09    2
1.976427e+09    2
3.014402e+09    1
3.223832e+09    1
1.947319e+09    1
2.984689e+09    1
Name: count, dtype: int64

In [12]:
pagamento_maior_max["mes"].value_counts()

mes
12    10
9      7
7      6
10     6
5      5
3      3
6      3
8      2
2      2
4      1
11     1
Name: count, dtype: int64

### Lei de Benford 

In [13]:
df_benford = df.copy()
display(df_benford.head(2))

,ano,mes,fase,codigo_unidade,unidade,valor,codigo_credor,credor_nome,codigo_funcao,codigo_natureza,cod_grupo_despesa
0,2025,1,PAGAMENTO,110103,Procuradoria Geral do Estado,164832.0,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",3.0,333903916,3
1,2025,1,PAGAMENTO,110103,Procuradoria Geral do Estado,162405.0,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",3.0,333903916,3


In [14]:
nome_colunas = df_benford.columns
for colunas in nome_colunas:
    print(f"Coluna: {colunas}")

Coluna: ano
Coluna: mes
Coluna: fase
Coluna: codigo_unidade
Coluna: unidade
Coluna: valor
Coluna: codigo_credor
Coluna: credor_nome
Coluna: codigo_funcao
Coluna: codigo_natureza
Coluna: cod_grupo_despesa


In [15]:
df_benford = df_benford.drop(["ano","unidade","codigo_unidade","codigo_funcao","codigo_natureza","cod_grupo_despesa"],axis=1)

#[["mes","codigo_credor","fase","credor_nome","valor"]]

In [16]:
df_benford = df_benford[["mes","codigo_credor","credor_nome","valor"]]
df_benford

,mes,codigo_credor,credor_nome,valor
0,1,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",164832.0
1,1,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",162405.0
2,1,14151000000288,C & S VIGILÃNCIA E SEGURANÃA PATRIMONIAL EIRELI,432708.0
3,1,23361040000164,CASTELUCCI EMPREENDIMENTOS E SERVIÃOS GERAIS ...,20402.0
4,1,00000PF0000031,INDENIZACAO VALE TRANSPORTE,1092.0
...,...,...,...,...
506455,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,71592.0
506456,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,10768794.0
506457,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,3134933.0
506458,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,1626707.0


In [17]:
linhas_df, colunas_df = df_benford.shape
print(f"A base que será aplicada a lei de benford tem: {linhas_df} Linhas e {colunas_df} colunas")

A base que será aplicada a lei de benford tem: 506460 Linhas e 4 colunas


Seja:

$$
V = \{v_1, v_2, v_3, \ldots, v_n\}
$$

onde \(V\) representa o vetor formado pelos valores da coluna `valor` do DataFrame `df_benford`.


In [18]:
valores = df_benford['valor'].values

### Normalização dos Valores

Para cada valor $$V_i$$, realiza-se a normalização para o intervalo \([1,10)$$, preservando apenas os dígitos significativos iniciais:

$$
N_i = \frac{V_i}{10^{\lfloor \log_{10}(V_i) \rfloor}}
$$

Onde:

- $V_i$ representa o valor original da $i$-ésima observação;
- $\log_{10}(V_i)$ representa o logaritmo na base 10 de $V_i$;
- $N_i$ representa o valor normalizado obtido a partir de $V_i$.

Exemplos:

| Valor $$V_i$$ | Valor Normalizado $$N_i$$ |
|-----------------|----------------------------|
| 1234 | 1,234 |
| 5678 | 5,678 |
| 98 | 9,8 |
| 456 | 4,56 |

In [19]:
norm = valores / 10 ** np.floor(np.log10(valores))
print(norm)

[1.64832  1.62405  4.32708  ... 3.134933 1.626707 7.06131 ]


### Extração dos Dígitos Significativos

Após a normalização dos valores, os dígitos significativos são obtidos por:

$$
D_k = \left\lfloor \left( N \times 10^{k-1} \right) \bmod 10 \right\rfloor
$$

Onde:

- \(D_k\): k-ésimo dígito significativo;
- \(N\): valor normalizado;
- \(k\): posição do dígito (\(k=1,2,3,4\));
- \(\bmod\): operador módulo;
- \(\lfloor \cdot \rfloor\): função piso (*floor*).


In [20]:
for k, nome in enumerate(
    ["primeiro_digito","segundo_digito","terceiro_digito", "quarto_digito"],
    start=1
):
    df_benford[nome] = np.floor((norm * 10**(k-1)) % 10).astype("uint8")

display(df_benford)

,mes,codigo_credor,credor_nome,valor,primeiro_digito,segundo_digito,terceiro_digito,quarto_digito
0,1,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",164832.0,1,6,4,8
1,1,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",162405.0,1,6,2,4
2,1,14151000000288,C & S VIGILÃNCIA E SEGURANÃA PATRIMONIAL EIRELI,432708.0,4,3,2,7
3,1,23361040000164,CASTELUCCI EMPREENDIMENTOS E SERVIÃOS GERAIS ...,20402.0,2,0,4,0
4,1,00000PF0000031,INDENIZACAO VALE TRANSPORTE,1092.0,1,0,9,2
...,...,...,...,...,...,...,...,...
506455,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,71592.0,7,1,5,9
506456,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,10768794.0,1,0,7,6
506457,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,3134933.0,3,1,3,4
506458,12,46777118000121,SECRETARIA DE ESTADO DA PESCA E AQUICULTURA DO...,1626707.0,1,6,2,6


#### Primeiro Digito

### Primeiro Dígito Significativo

O primeiro dígito significativo é obtido pela parte inteira do valor normalizado:

$$
D_1 = \left\lfloor N \right\rfloor
$$

onde:

$$
N = \frac{V}{10^{\lfloor \log_{10}(V) \rfloor}}
$$

Substituindo a expressão de \(N\):

$$
D_1 =
\left\lfloor
\frac{V}
{10^{\lfloor \log_{10}(V) \rfloor}}
\right\rfloor
$$

Exemplo:

$$
V = 1234
$$

$$
N = \frac{1234}{10^3} = 1,234
$$

$$
D_1 = \lfloor 1,234 \rfloor = 1
$$

In [28]:
df_benford_1d = df_benford[["mes","codigo_credor","credor_nome","valor","primeiro_digito"]]
display(df_benford_1d.head(3))

,mes,codigo_credor,credor_nome,valor,primeiro_digito
0,1,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",164832.0,1
1,1,29485582000125,"FP- PROJETOS, GERENCIAMENTOS, SERVIÃOS E EMPR...",162405.0,1
2,1,14151000000288,C & S VIGILÃNCIA E SEGURANÃA PATRIMONIAL EIRELI,432708.0,4


##### Frequência Observada dos Primeiros Dígitos

A frequência observada de cada primeiro dígito é calculada por:

$$
F(d) = \sum_{i=1}^{n} I(D_i = d)
$$

Onde:

- \(F(d)\): frequência observada do dígito \(d\);
- \(D_i\): primeiro dígito da i-ésima observação;
- \(n\): número total de registros;
- \(I(\cdot)\): função indicadora.

A função indicadora é definida como:

$$
I(D_i=d)=
\begin{cases}
1, & \text{se } D_i=d\\
0, & \text{caso contrário}
\end{cases}
$$

In [22]:
frequencia_real = (
    df_benford_1d["primeiro_digito"]
    .value_counts()
    .sort_index()
)

display(frequencia_real)

primeiro_digito
1    137231
2     89384
3     63060
4     46036
5     43593
6     38976
7     44996
8     19346
9     23838
Name: count, dtype: int64

$$
n_{d1} = \sum_{d=1}^{9} f_d
$$

Onde:

- $n_{d1}$: número total de observações analisadas;
- $f_d$: frequência observada do dígito $d$;
- $d \in \{1,2,\ldots,9\}$.

In [23]:
n_d1 = frequencia_real.sum()
display(n_d1)

#OU
#n_d1, colunas_1d = df_benford_1d.shape


np.int64(506460)

$$
p_d = \frac{f_d}{n_{d1}}
$$

In [24]:
percentual_real = frequencia_real / n_d1
display(percentual_real)

primeiro_digito
1    0.270961
2    0.176488
3    0.124511
4    0.090898
5    0.086074
6    0.076958
7    0.088844
8    0.038198
9    0.047068
Name: count, dtype: float64

$$
D = \{1,2,3,4,5,6,7,8,9\}
$$

Onde:

- $D$: conjunto dos possíveis primeiros dígitos significativos.

In [25]:
digitos = np.arange(1,10)
display(digitos)

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

$$
B_d = \log_{10}\left(1 + \frac{1}{d}\right)
$$

Onde:

- $B_d$: probabilidade teórica do dígito $d$ segundo a Lei de Benford;
- $d$: primeiro dígito significativo, com $d \in \{1,2,\ldots,9\}$;
- $\log_{10}$: logaritmo na base 10.

In [26]:
percentual_benford = np.log10(1+1/digitos)
display(percentual_benford)

array([0.30103   , 0.17609126, 0.12493874, 0.09691001, 0.07918125,
       0.06694679, 0.05799195, 0.05115252, 0.04575749])

$$
E_d = B_d \cdot n_{d1}
$$

$$
E_d = \log_{10}\left(1+\frac{1}{d}\right)\cdot n_{d1}
$$

Onde:

- $E_d$: frequência esperada do dígito $d$;
- $B_d$: probabilidade teórica do dígito $d$ segundo a Lei de Benford;
- $n_{d1}$: número total de observações analisadas.

In [27]:
frequencia_benford = percentual_benford * n_d1
display(frequencia_benford)

array([152459.65160398,  89183.17906134,  63276.47254264,  49081.04518806,
        40102.13387328,  33905.87107632,  29370.60146632,  25906.7065187 ,
        23174.33866936])